# Réplica ampliada: Kesgin et al. (2025)
Réplica del flujo del artículo *Beyond Performance* usando los dos archivos de UCI (matemática y portugués, 33 variables), escalado, SMOTE-NC y métricas de exactitud, F1 y AUC. Se agrega **CatBoost** como cuarto modelo.

Diferencias con el artículo: la equidad se mide a mano (paridad demográfica y probabilidades igualadas por sexo) y no se aplica debiasing adversarial. Los hiperparámetros son los que indica el artículo.

Subir antes `student-mat.csv` y `student-por.csv` a la carpeta de Colab.

In [ ]:
!pip install -q imbalanced-learn shap lime catboost xgboost

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTENC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import shap, lime, lime.lime_tabular
SEED = 42

## 1. Datos (matemática + portugués)

In [ ]:
mat = pd.read_csv('student-mat.csv', sep=';')
por = pd.read_csv('student-por.csv', sep=';')
data = pd.concat([mat, por], ignore_index=True)
print(mat.shape, por.shape, data.shape)   # esperado: (395, 33) (649, 33) (1044, 33)
# Nota: ~382 alumnos aparecen en ambos archivos; puede haber el mismo alumno en entrenamiento y prueba.
y = (data['G3'] >= 10).astype(int)          # aprueba = 1
X = data.drop(columns=['G3'])
print(y.value_counts())

## 2. Preprocesamiento: one-hot, partición 80/20 estratificada, escalado, SMOTE-NC (solo entrenamiento)

In [ ]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

cat_idx = [X_train.columns.get_loc(c) for c in X_train.columns if c not in num_cols]
smote = SMOTENC(categorical_features=cat_idx, random_state=SEED)
X_res, y_res = smote.fit_resample(X_train, y_train)
print('Antes:', y_train.value_counts().to_dict(), '| Después:', pd.Series(y_res).value_counts().to_dict())

## 3. Modelos (hiperparámetros del artículo) + CatBoost

In [ ]:
models = {
    'Regresión logística': LogisticRegression(penalty='l2', max_iter=1000, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_split=5, random_state=SEED),
    'XGBoost': XGBClassifier(learning_rate=0.1, n_estimators=200, max_depth=5, eval_metric='logloss', random_state=SEED),
    'CatBoost': CatBoostClassifier(iterations=200, depth=5, learning_rate=0.1, random_seed=SEED, verbose=0),
}
for m in models.values():
    m.fit(X_res, y_res)

## 4. Evaluación en el conjunto de prueba

In [ ]:
rows = {}
for name, m in models.items():
    pred = m.predict(X_test); prob = m.predict_proba(X_test)[:, 1]
    rows[name] = {'Exactitud': accuracy_score(y_test, pred), 'F1': f1_score(y_test, pred), 'AUC': roc_auc_score(y_test, prob)}
res = pd.DataFrame(rows).T.round(3)
print(res)
res.to_csv('resultados_modelos.csv')

## 5. Explicabilidad: SHAP (XGBoost) y LIME (regresión logística)

In [ ]:
sv = shap.TreeExplainer(models['XGBoost']).shap_values(X_test)
shap.summary_plot(sv, X_test, plot_type='bar', max_display=10)

In [ ]:
lime_exp = lime.lime_tabular.LimeTabularExplainer(X_res.values, feature_names=list(X_res.columns),
                                                  class_names=['Reprueba','Aprueba'], mode='classification')
exp = lime_exp.explain_instance(X_test.iloc[0].values, models['Regresión logística'].predict_proba, num_features=8)
exp.show_in_notebook()

## 6. Equidad por sexo (mujeres vs. hombres)

In [ ]:
def equidad(modelo):
    pred = modelo.predict(X_test)
    h = (X_test['sex_M'] == 1).values; m = ~h
    dp = abs(pred[h].mean() - pred[m].mean())
    def tpr(g): return pred[g & (y_test.values == 1)].mean()
    def fpr(g): return pred[g & (y_test.values == 0)].mean()
    eo = max(abs(tpr(h) - tpr(m)), abs(fpr(h) - fpr(m)))
    return round(dp, 3), round(eo, 3)
eq = pd.DataFrame({n: equidad(m) for n, m in models.items()}, index=['Dif. paridad demográfica', 'Dif. probab. igualadas']).T
print(eq)
eq.to_csv('equidad_modelos.csv')